# exactory-lab colab runner

Compute only, never an LLM. This notebook borrows a Colab GPU to run
experiment scripts that `exactory-lab run --backend colab` ships through a
shared Google Drive folder. Ideation, writing, and review stay with the
agent on your machine.

Setup, once:

1. Runtime → Change runtime type → a GPU.
2. Run the cell below and approve the Drive mount.
3. On your machine, point `EXACTORY_LAB_COLAB_DIR` at the local mirror of
   `My Drive/exactory-colab` (Google Drive for Desktop syncs it).

The cell loops until you interrupt it: it writes a heartbeat, claims each
READY job exactly once, runs the script, and writes the results folder with
DONE last. Scripts generate their own small or synthetic data; no datasets
travel through the folder.

In [ ]:
import json, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
SYNC_ROOT = Path("/content/drive/MyDrive/exactory-colab")
SYNC_ROOT.mkdir(parents=True, exist_ok=True)
POLL_SECONDS = 10.0


def parse_metric(stdout_text):
    metric = None
    for line in stdout_text.splitlines():
        line = line.strip()
        if not (line.startswith("{") and line.endswith("}")):
            continue
        try:
            candidate = json.loads(line)
        except ValueError:
            continue
        if isinstance(candidate, dict) and "metric" in candidate:
            metric = candidate
    return metric


def process_job(job_dir):
    job = json.loads((job_dir / "job.json").read_text())
    work_dir = SYNC_ROOT / "_work" / job["job_id"]
    if work_dir.exists():
        shutil.rmtree(work_dir)
    shutil.copytree(job_dir / "code", work_dir / "code")
    for dir_name in ("logs", "results", "plots"):
        (work_dir / dir_name).mkdir(parents=True, exist_ok=True)
    env = dict(__import__("os").environ)
    if job.get("seed") is not None:
        env["EXACTORY_LAB_SEED"] = str(job["seed"])
    start = time.monotonic()
    timed_out = False
    try:
        completed = subprocess.run(
            [sys.executable, str(work_dir / job["script"])],
            cwd=str(work_dir), env=env, capture_output=True, text=True,
            timeout=float(job["timeout"]),
        )
        returncode, stdout_text, stderr_text = (
            completed.returncode, completed.stdout, completed.stderr
        )
    except subprocess.TimeoutExpired as error:
        timed_out, returncode = True, -1
        stdout_text = error.stdout or ""
        stderr_text = (error.stderr or "") + "\n[runner] timeout"
    duration_s = round(time.monotonic() - start, 3)
    metric = parse_metric(stdout_text)
    node = job["node"]
    if metric is None:
        node_results = work_dir / "results" / f"{node}.json"
        if node_results.is_file():
            try:
                metric = json.loads(node_results.read_text())
            except ValueError:
                metric = None
    is_buggy = returncode != 0 or timed_out or metric is None
    record = {
        "node": node, "ok": not is_buggy, "is_buggy": is_buggy,
        "returncode": returncode, "timed_out": timed_out,
        "duration_s": duration_s, "metric": metric, "seed": job.get("seed"),
        "log": f"experiment/logs/{node}.log",
        "stderr_tail": stderr_text[-800:], "backend": "colab",
    }
    (work_dir / "logs" / f"{node}.log").write_text(
        f"$ {sys.executable} {job['script']}\n# rc={returncode}"
        f" duration={duration_s}s\n\n----- STDOUT -----\n{stdout_text}\n"
        f"----- STDERR -----\n{stderr_text}\n"
    )
    (work_dir / "results" / f"{node}.json").write_text(
        json.dumps(record, indent=2) + "\n"
    )
    result_dir = SYNC_ROOT / "results" / job["job_id"]
    for dir_name in ("logs", "results", "plots"):
        target_dir = result_dir / dir_name
        target_dir.mkdir(parents=True, exist_ok=True)
        for source_file in (work_dir / dir_name).iterdir():
            shutil.copy2(source_file, target_dir / source_file.name)
    (result_dir / "DONE").write_text(str(time.time()))
    print(f"[done] {job['job_id']} ok={record['ok']} {duration_s}s")


print(f"[runner] serving {SYNC_ROOT}")
while True:
    (SYNC_ROOT / "RUNNER_ALIVE").write_text(str(time.time()))
    jobs_dir = SYNC_ROOT / "jobs"
    if jobs_dir.is_dir():
        for ready_path in sorted(jobs_dir.glob("*/READY")):
            job_dir = ready_path.parent
            if (job_dir / "PROCESSED").exists():
                continue
            (job_dir / "PROCESSED").write_text(str(time.time()))
            try:
                process_job(job_dir)
            except Exception as error:
                print(f"[error] {job_dir.name}: {error}")
    time.sleep(POLL_SECONDS)